In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [60]:
%%writefile src/cpbl.py
import requests
import pandas as pd
from bs4 import BeautifulSoup

half_codes={'上半季':'1','下半季':'2'}

def fetch_cpbl_standings(season_code,refresh=True):
    path_way=Path(__file__).resolve().parent.parent/"data"/"raw"
    ua='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
    path=path_way/f'cpbl_standing_{season_code}.html'
    if path.exists() and not refresh:
        return path.read_text(encoding="utf-8")
    session=requests.Session()
    page=session.get(
        'https://www.cpbl.com.tw/standings/season',
        headers={
            'User-agent':ua
        },
        timeout=20
    )
    soup=BeautifulSoup(page.text,'html.parser')
    cookie=soup.find('input',{'name':'__RequestVerificationToken'})
    form_cookie=cookie.get('value','')
    res=session.post(
        'https://www.cpbl.com.tw/standings/seasonaction',
        headers={
            'User-agent':ua,
            'content-type':'application/x-www-form-urlencoded; charset=UTF-8',
            'x-requested-with':'XMLHttpRequest',
            'origin':'https://www.cpbl.com.tw',
            'referer':'https://www.cpbl.com.tw/standings/season',
        },
        data={
            '__RequestVerificationToken':form_cookie,
            'ExecAction':'',
            'IndexOfPages':'0',
            'Kindcode':'A',
            'SeasonCode': season_code,
            'GameEndDate':''
        }
    )
    res.status_code()
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(res.text,encoding="utf-8")
    return res.text

def cpbl_whole_half_standing(refresh=True):
    ua='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
    session=requests.Session()
    page=session.get(
        'https://www.cpbl.com.tw/standings/season',
        headers={
            'User-agent':ua
        },
        timeout=20
    )
    soup=BeautifulSoup(page.text,'html.parser')
    cookie=soup.find('input',{'name':'__RequestVerificationToken'})
    form_cookie=cookie.get('value','')
    res=session.post(
        'https://www.cpbl.com.tw/standings/seasonaction',
        headers={
            'User-agent':ua,
            'content-type':'application/x-www-form-urlencoded; charset=UTF-8',
            'x-requested-with':'XMLHttpRequest',
            'origin':'https://www.cpbl.com.tw',
            'referer':'https://www.cpbl.com.tw/standings/season',
        },
        data={
            '__RequestVerificationToken':form_cookie,
            'ExecAction':'',
            'IndexOfPages':'0',
            'Kindcode':'A',
            'SeasonCode': '1',
            'GameEndDate':''
        }
    )
    res2=BeautifulSoup(res.text,'html.parser')
    table=res2.select_one('table')
    team_score_data_top=[]
    for tr in table.select('tr'):
        num_td=tr.select('td.num')
        name_id=tr.select＿one('a')
        if name_id and len(num_td)>=14:
            team_name=name_id.text.strip()
            win=num_td[1].text.strip().split('-')[0]
            lose=num_td[1].text.strip().split('-')[1]
            tie=num_td[1].text.strip().split('-')[2]
            team_score_data_top.append({
                'team_name':team_name,
                '勝':win,
                '敗':lose,
                '平手':tie
            })
    df_team_score_top=pd.DataFrame(team_score_data_top).set_index('team_name')
    
    session1=requests.Session()
    page1=session1.get(
        'https://www.cpbl.com.tw/standings/season',
        headers={
            'User-agent':ua
        },
        timeout=20
    )
    soup1=BeautifulSoup(page1.text,'html.parser')
    cookie1=soup1.find('input',{'name':'__RequestVerificationToken'})
    form_cookie1=cookie1.get('value','')
    res1=session1.post(
        'https://www.cpbl.com.tw/standings/seasonaction',
        headers={
            'User-agent':ua,
            'content-type':'application/x-www-form-urlencoded; charset=UTF-8',
            'x-requested-with':'XMLHttpRequest',
            'origin':'https://www.cpbl.com.tw',
            'referer':'https://www.cpbl.com.tw/standings/season',
        },
        data={
            '__RequestVerificationToken':form_cookie,
            'ExecAction':'',
            'IndexOfPages':'0',
            'Kindcode':'A',
            'SeasonCode': '2',
            'GameEndDate':''
        }
    )
    res12=BeautifulSoup(res1.text,'html.parser')
    table1=res12.select_one('table')
    team_score_data_under=[]
    for tr in table1.select('tr'):
        num_td=tr.select('td.num')
        name_id=tr.select＿one('a')
        if name_id and len(num_td)>=14:
            team_name=name_id.text.strip()
            win=num_td[1].text.strip().split('-')[0]
            lose=num_td[1].text.strip().split('-')[1]
            tie=num_td[1].text.strip().split('-')[2]
            team_score_data_under.append({
                'team_name':team_name,
                '勝':win,
                '敗':lose,
                '平手':tie
            })
    df_team_score_under=pd.DataFrame(team_score_data_under).set_index('team_name')
    df_team_score_whole=pd.concat([df_team_score_top,df_team_score_under],ignore_index=False,axis=0,keys=['上半季','下半季'])
    return df_team_score_whole

Overwriting src/cpbl.py


In [74]:
%%writefile -a src/cpbl.py
import requests
import json
from bs4 import BeautifulSoup

def rebas_cpbl_half_standing(section):
    session=requests.Session()
    ua='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
    page=session.get(
        f'https://www.rebas.tw/api/seasons/CPBL-2026-oB/leaders?type=team&section={section}&pa=undefined',
        headers={
            'User-agent':ua
        }
    )
    page.status_code
    data=page.json()['data']
    row=[]
    for i in data:
        team_name=i['team']['name']
        total_game=i['total']
        total_win=i['wins']
        total_lose=i['loses']
        total_draw=i['draws']
        win_pct=i['PCT']
        diff=i['DIFF']
        row.append({
            '球隊':team_name,
            '場數':total_game,
            '勝':total_win,
            '敗':total_lose,
            '平手':total_draw,
            '勝率':win_pct,
            '得失分差':diff
        })
    df_standing=pd.DataFrame(row).set_index('球隊')
    return df_standing

Appending to src/cpbl.py


In [76]:
!git add .
!git commit -m "Day 5: 中職官網半季戰績與交叉比對"

[main 86ed225] Day 5: 中職官網半季戰績與交叉比對
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 7 files changed, 788 insertions(+), 61 deletions(-)
 create mode 100644 D04.ipynb
 create mode 100644 src/cpbl.py
